In [ ]:
import pandas as pd
import numpy as np
import math
import warnings
import pickle
import re
import scipy.stats
import statsmodels.api as sm
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.patches as mpatches
from pathlib import Path
from matplotlib.lines import Line2D
from matplotlib.legend_handler import HandlerBase
from matplotlib.patches import Patch
from typing import Dict, List, Optional, Tuple
from tqdm import tqdm

# Define functions

In [ ]:
def run_model(
    x_col: str,
    *,
    time_col: str = "DateTime",
    n_boot: int = 500,
    seed: int = 42,
    add_entity_fe: bool = True,
    add_month_fe: bool = True,
    fs_entity_fe: bool = False,
    fs_month_fe: bool = False,
    squared_instruments: bool = False,
    quadratic_headroom: bool = False,
    bin_dummies: bool = False,
    n_bins: int = 10,
    bin_reference: str = "first",
    add_lags: int = 1,
    plain_logit: bool = False,
    leave_out: str = None,
):

    g = globals()

    # Functional-form sanity checks
    if quadratic_headroom and bin_dummies:
        raise ValueError(
            "quadratic_headroom and bin_dummies are mutually exclusive — "
            "they're alternative ways to capture nonlinearity."
        )

    if (quadratic_headroom or bin_dummies) and "congestion" in x_col:
        raise ValueError(
            "Polynomial/bin expansion requires a continuous treatment (e.g. headroom). "
            "For binary congestion, the polynomial/bin transformation is degenerate."
        )

    if bin_dummies and n_bins < 2:
        raise ValueError("n_bins must be >= 2.")

    # ------------------------------------------------------------------
    # 1. Pick df and y
    # ------------------------------------------------------------------
    if x_col.startswith("ex_"):
        df = g["df_mod_pos_with_ee"].copy()
        y  = pd.Series(g["y_pos"], index=df.index).astype(int)
    elif x_col.startswith("im_"):
        df = g["df_mod_neg_with_ee"].copy()
        y  = pd.Series(g["y_neg"], index=df.index).astype(int)
    else:
        raise ValueError("x_col must start with 'im_' or 'ex_'.")

    df[time_col] = pd.to_datetime(df[time_col])

    entity_cols = [c for c in df.columns if c.startswith("entity_")]
    if not entity_cols:
        raise ValueError("No entity_ columns found — cannot cluster by entity.")

    # ------------------------------------------------------------------
    # 2. Squared columns
    # ------------------------------------------------------------------
    if squared_instruments:
        df["wind_sl_sq"]          = df["wind_sl"]          ** 2
        df["solar_sl_sq"]         = df["solar_sl"]         ** 2
        df["load_sl_sq"] = df["load_sl"] ** 2
        df["wind_sq"]             = df["wind"]              ** 2
        df["solar_sq"]            = df["solar"]             ** 2

    # ------------------------------------------------------------------
    # 3. Lags
    # ------------------------------------------------------------------
    base_lag_vars = ["wind", "wind_sl", "solar", "solar_sl",
                     "load", "load_sl"]
    if squared_instruments:
        base_lag_vars += ["wind_sq", "solar_sq",
                          "wind_sl_sq", "solar_sl_sq", "load_sl_sq"]

    entity_id = df[entity_cols].idxmax(axis=1)
    df = df.assign(_eid=entity_id).sort_values(["_eid", time_col])

    for v in base_lag_vars:
        for lag in range(1, add_lags + 1):
            df[f"L{lag}_{v}"] = df.groupby("_eid")[v].shift(lag)

    # ------------------------------------------------------------------
    # 4. Column lists
    # ------------------------------------------------------------------
    Z_cols_base = ["wind_sl", "solar_sl", "load_sl"]
    Z_sq_cols   = (["wind_sl_sq", "solar_sl_sq", "load_sl_sq"]
                   if squared_instruments else [])
    all_Z_cols  = Z_cols_base + Z_sq_cols

    W_cols = ["wind", "solar", "load"]

    for lag in range(1, add_lags + 1):
        W_cols += [
            f"L{lag}_wind",
            f"L{lag}_wind_sl",
            f"L{lag}_solar",
            f"L{lag}_solar_sl",
            f"L{lag}_load",
            f"L{lag}_load_sl",
        ]

    W_cols += [
        "carbon",
        "gas",
        "coal",
    ]

    if squared_instruments:
        W_cols += ["wind_sq", "solar_sq"]
        for lag in range(1, add_lags + 1):
            W_cols += [
                f"L{lag}_wind_sq",
                f"L{lag}_solar_sq",
                f"L{lag}_wind_sl_sq",
                f"L{lag}_solar_sl_sq",
                f"L{lag}_load_sl_sq",
            ]

    if add_entity_fe:
        W_cols += entity_cols
    if add_month_fe:
        W_cols += [c for c in df.columns if c.startswith("month_")]

    W_cols_fs = [c for c in W_cols if not (
        (not fs_entity_fe and c.startswith("entity_")) or
        (not fs_month_fe  and c.startswith("month_"))
    )]

    # ------------------------------------------------------------------
    # 5. Build all_data
    # ------------------------------------------------------------------
    keep = list(dict.fromkeys([x_col, time_col] + W_cols + all_Z_cols))
    all_data = (pd.concat([df[keep], y.rename("y")], axis=1)
                  .dropna()
                  .reset_index(drop=True))

    all_data["_entity_id"] = all_data[entity_cols].idxmax(axis=1).values

    # ---- Leave-one-out: drop a single entity by EIC code ----
    if leave_out is not None:
        target_col = next((c for c in entity_cols if leave_out in c), None)
        if target_col is None:
            raise ValueError(
                f"leave_out='{leave_out}' not found among entity columns. "
                f"Available (sample): {[c.replace('entity_','') for c in entity_cols[:5]]}..."
            )
        keep_mask = all_data["_entity_id"] != target_col
        n_drop = int((~keep_mask).sum())
        all_data = all_data[keep_mask].reset_index(drop=True)
        print(f"[leave_out] Dropped entity {leave_out} ({target_col}): "
              f"{n_drop} obs removed, {len(all_data)} remain.")
        # Remove the now all-zero dummy from the design
        all_data = all_data.drop(columns=[target_col])
        entity_cols = [c for c in entity_cols if c != target_col]
        W_cols = [c for c in W_cols if c != target_col]
        W_cols_fs = [c for c in W_cols_fs if c != target_col]

    # Polynomial treatment column
    x_sq_name = f"{x_col}_sq"

    if quadratic_headroom:
        all_data[x_sq_name] = all_data[x_col].astype(float) ** 2

    # ------------------------------------------------------------------
    # Bin dummy construction
    # ------------------------------------------------------------------
    bin_edges = None
    bin_names = []
    bin_ref_idx = None

    if bin_dummies:
        x_full = all_data[x_col].astype(float).values

        # Quantile-based edges, from observed x
        edges_q = np.quantile(x_full, np.linspace(0, 1, n_bins + 1))
        # Make edges strictly increasing (handle ties at extremes)
        edges_q = np.unique(edges_q)
        if len(edges_q) - 1 < n_bins:
            print(f"[Warning] n_bins={n_bins} requested but only "
                  f"{len(edges_q)-1} distinct quantile edges available. "
                  f"Using {len(edges_q)-1} bins instead.")
            n_bins_eff = len(edges_q) - 1
        else:
            n_bins_eff = n_bins

        # Make endpoints inclusive
        edges_q[0]  -= 1e-9
        edges_q[-1] += 1e-9

        bin_edges = edges_q

        # Bin assignment for full sample
        bin_idx = np.digitize(x_full, edges_q[1:-1], right=False)  # 0 .. n_bins-1
        bin_idx = np.clip(bin_idx, 0, n_bins_eff - 1)

        # Generate bin names with the bin range for readability
        bin_names = []
        for i in range(n_bins_eff):
            lo = edges_q[i] + (1e-9 if i == 0 else 0)
            hi = edges_q[i + 1] - (1e-9 if i == n_bins_eff - 1 else 0)
            bin_names.append(f"{x_col}_bin{i+1}_[{lo:.2f},{hi:.2f}]")

        # Pick reference bin
        if bin_reference == "first":
            bin_ref_idx = 0
        elif bin_reference == "last":
            bin_ref_idx = n_bins_eff - 1
        elif isinstance(bin_reference, (int, np.integer)):
            if not (0 <= bin_reference < n_bins_eff):
                raise ValueError(
                    f"bin_reference={bin_reference} out of range [0, {n_bins_eff-1}]"
                )
            bin_ref_idx = int(bin_reference)
        else:
            raise ValueError("bin_reference must be 'first', 'last', or an integer.")

        # Build dummy columns (all bins, including reference — we'll drop ref later)
        for i in range(n_bins_eff):
            all_data[bin_names[i]] = (bin_idx == i).astype(float)

        bin_size_summary = pd.Series({
            bin_names[i]: int((bin_idx == i).sum()) for i in range(n_bins_eff)
        })
        print(f"\nBin dummies ({n_bins_eff} bins, "
              f"reference: {bin_names[bin_ref_idx]}):")
        for nm, sz in bin_size_summary.items():
            tag = "  <-- ref" if nm == bin_names[bin_ref_idx] else ""
            print(f"  {nm}: n={sz}{tag}")

        # Names of dummies entering the regression (omit reference)
        bin_active_names = [nm for i, nm in enumerate(bin_names)
                            if i != bin_ref_idx]
    else:
        bin_active_names = []
        n_bins_eff = 0

    # Labels
    if plain_logit:
        fs_label     = "None (no first stage)"
        method_label = "PLAIN LOGIT"
    else:
        fs_label     = "OLS"
        method_label = "2SRI LOGIT"

    if quadratic_headroom:
        poly_label = "quadratic (x + x²)"
    elif bin_dummies:
        poly_label = f"bin dummies ({n_bins_eff} quantile bins, omit bin {bin_ref_idx+1})"
    else:
        poly_label = "linear (x)"

    print(f"all_data shape: {all_data.shape}")
    print(f"Unique entities (clusters): {all_data['_entity_id'].nunique()}")
    if not plain_logit:
        print(f"Instruments: {all_Z_cols}")
    print(f"Estimation method: {method_label}")
    print(f"First stage: {fs_label}")
    print(f"Functional form: {poly_label}")
    if not plain_logit:
        print(f"FS entity FE: {fs_entity_fe}  |  FS month FE: {fs_month_fe}")
        print(f"FS regressors: {len(W_cols_fs)}  |  2nd-stage regressors: {len(W_cols)}")
    else:
        print(f"2nd-stage regressors: {len(W_cols)}")
    print(f"add_lags: {add_lags}")

    entity_index = {
        e: np.where(all_data["_entity_id"].values == e)[0]
        for e in all_data["_entity_id"].unique()
    }

    y_arr    = all_data["y"].values.astype(int)
    x_arr    = all_data[x_col].values.astype(float)
    W_arr    = all_data[W_cols].values.astype(float)
    W_fs_arr = all_data[W_cols_fs].values.astype(float) if not plain_logit else None
    Z_arr    = all_data[all_Z_cols].values.astype(float) if not plain_logit else None

    # Build struct_names conditionally
    struct_names = []
    if bin_dummies:
        struct_names += bin_active_names
    else:
        struct_names.append(x_col)
        if quadratic_headroom:
            struct_names.append(x_sq_name)

    if not plain_logit:
        struct_names.append("vhat")

    # ------------------------------------------------------------------
    # Helper: compute bin dummies from a given x array using fixed edges
    # ------------------------------------------------------------------
    def _bin_matrix(x_vals, edges, ref_idx, n_b):
        """Return matrix of bin dummies (n × n_b-1), omitting reference bin."""
        idx = np.digitize(x_vals, edges[1:-1], right=False)
        idx = np.clip(idx, 0, n_b - 1)
        cols = []
        for i in range(n_b):
            if i == ref_idx:
                continue
            cols.append((idx == i).astype(float))
        return np.column_stack(cols) if cols else np.zeros((len(x_vals), 0))

    # ------------------------------------------------------------------
    # 6. Bootstrap fitter
    # ------------------------------------------------------------------
    def _both_stages_boot(y_, x_, W_, W_fs_, Z_):
        keep_mask = W_.std(axis=0) > 0
        W_        = W_[:, keep_mask]
        n_w       = W_.shape[1]

        # ---- PLAIN LOGIT ----
        if plain_logit:
            col_names = ["const"] + [f"w{i}" for i in range(n_w)]
            X2_cols   = [np.ones(len(W_)), W_]

            if bin_dummies:
                bin_mat = _bin_matrix(x_, bin_edges, bin_ref_idx, n_bins_eff)
                col_names += bin_active_names
                X2_cols.append(bin_mat)
            else:
                col_names.append(x_col)
                X2_cols.append(x_)
                if quadratic_headroom:
                    col_names.append(x_sq_name)
                    X2_cols.append(x_ ** 2)

            X2 = pd.DataFrame(np.column_stack(X2_cols), columns=col_names)

            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                ss = sm.Logit(y_, X2.astype(float)).fit(
                    disp=0, maxiter=200, cov_type="nonrobust"
                )
            return ss.params[struct_names].values

        # ---- 2SRI: first stage (OLS) ----
        keep_fs = W_fs_.std(axis=0) > 0
        W_fs_   = W_fs_[:, keep_fs]

        X1 = np.column_stack([np.ones(len(W_fs_)), W_fs_, Z_])

        beta1, _, _, _ = np.linalg.lstsq(X1, x_, rcond=None)
        vhat = x_ - X1 @ beta1

        # ---- 2SRI: second stage ----
        col_names = ["const"] + [f"w{i}" for i in range(n_w)]
        X2_cols   = [np.ones(len(W_)), W_]

        if bin_dummies:
            # Bins from observed x (CF correction handles endogeneity)
            bin_mat = _bin_matrix(x_, bin_edges, bin_ref_idx, n_bins_eff)
            col_names += bin_active_names
            X2_cols.append(bin_mat)
        else:
            col_names.append(x_col)
            X2_cols.append(x_)
            if quadratic_headroom:
                col_names.append(x_sq_name)
                X2_cols.append(x_ ** 2)

        col_names.append("vhat")
        X2_cols.append(vhat)

        X2 = pd.DataFrame(np.column_stack(X2_cols), columns=col_names)

        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            ss = sm.Logit(y_, X2.astype(float)).fit(
                disp=0, maxiter=200, cov_type="nonrobust"
            )

        return ss.params[struct_names].values

    # ------------------------------------------------------------------
    # 7. Full-sample estimation
    # ------------------------------------------------------------------
    x2_ser = all_data[x_col].astype(float)
    W2_sm  = sm.add_constant(all_data[W_cols].astype(float), has_constant="add")

    if plain_logit:
        fs        = None
        vhat_full = None
        f_joint = f_wind = f_solar = f_load = None
    else:
        X1_sm = sm.add_constant(
            pd.concat([all_data[W_cols_fs], all_data[all_Z_cols]], axis=1),
            has_constant="add"
        ).astype(float)

        fs = sm.OLS(all_data[x_col].astype(float), X1_sm).fit(
            cov_type="cluster",
            cov_kwds={"groups": all_data["_entity_id"]}
        )
        vhat_full = fs.resid

        f_joint_str = ", ".join([f"{z} = 0" for z in all_Z_cols])
        f_joint = fs.f_test(f_joint_str)
        f_wind  = fs.f_test("wind_sl = 0")
        f_solar = fs.f_test("solar_sl = 0")
        f_load  = fs.f_test("load_sl = 0")

    # Second stage
    if plain_logit:
        X2_parts = [W2_sm]
        if bin_dummies:
            X2_parts += [all_data[nm].rename(nm) for nm in bin_active_names]
        else:
            X2_parts.append(x2_ser.rename(x_col))
            if quadratic_headroom:
                X2_parts.append((x2_ser ** 2).rename(x_sq_name))

        X2_sm = pd.concat(X2_parts, axis=1)

        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            ss = sm.Logit(all_data["y"].astype(int), X2_sm.astype(float)).fit(
                disp=0, maxiter=200,
                cov_type="cluster",
                cov_kwds={"groups": all_data["_entity_id"]}
            )

    else:
        # 2SRI
        X2_parts = [W2_sm]
        if bin_dummies:
            X2_parts += [all_data[nm].rename(nm) for nm in bin_active_names]
        else:
            X2_parts.append(x2_ser.rename(x_col))
            if quadratic_headroom:
                X2_parts.append((x2_ser ** 2).rename(x_sq_name))
        X2_parts.append(vhat_full.rename("vhat"))

        X2_sm = pd.concat(X2_parts, axis=1)

        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            ss = sm.Logit(all_data["y"].astype(int), X2_sm.astype(float)).fit(
                disp=0, maxiter=200,
                cov_type="cluster",
                cov_kwds={"groups": all_data["_entity_id"]}
            )

    # Wald tests
    wald_xSq = wald_bins_joint = None

    if quadratic_headroom:
        exog_names = ss.model.exog_names
        j_sq = exog_names.index(x_sq_name)
        R_sq = np.zeros((1, len(exog_names)))
        R_sq[0, j_sq] = 1.0
        wald_xSq = ss.wald_test(R_sq, scalar=True)

    if bin_dummies and len(bin_active_names) > 0:
        # Joint test that all bin dummies = 0 (i.e. headroom has no effect)
        exog_names = ss.model.exog_names
        R_bins = np.zeros((len(bin_active_names), len(exog_names)))
        for r, nm in enumerate(bin_active_names):
            R_bins[r, exog_names.index(nm)] = 1.0
        wald_bins_joint = ss.wald_test(R_bins, scalar=True)

    full_struct = ss.params[struct_names].values

    # ------------------------------------------------------------------
    # Full-sample output
    # ------------------------------------------------------------------
    if not plain_logit:
        print(f"\n================ FIRST STAGE ({fs_label.upper()}) ================\n")
        print(fs.summary())
        print("\n--- Instrument diagnostics ---")
        for inst in all_Z_cols:
            coef_val = fs.params[inst]
            t_val    = fs.tvalues[inst]
            print(f"  {inst}: coef={coef_val:.4f}  t={t_val:.3f}")

        print(f"  Joint F(Z=0): F={float(np.asarray(f_joint.fvalue)):.3f}  "
              f"p={float(f_joint.pvalue):.4f}")

        if squared_instruments:
            f_sq_str = ", ".join([f"{z} = 0" for z in Z_sq_cols])
            f_sq = fs.f_test(f_sq_str)
            print(f"  Joint F(Z²=0): F={float(np.asarray(f_sq.fvalue)):.3f}  "
                  f"p={float(f_sq.pvalue):.4f}")

    print(f"\n================ SECOND STAGE ({method_label}) ================\n")
    print(ss.summary())
    if quadratic_headroom:
        print(f"\nWald test H0: {x_sq_name} = 0\n", wald_xSq)
    if bin_dummies and wald_bins_joint is not None:
        print(f"\nJoint Wald test H0: all bin coefficients = 0 (no effect of headroom)\n",
              wald_bins_joint)

    print("\n--- Structural params ---")
    for name, val in zip(struct_names, full_struct):
        print(f"  {name}: {val:.4f}")

    if n_boot == 0:
        print("\nn_boot=0: skipping bootstrap.")
        return _build_return(
            all_data, W2_sm, all_Z_cols, fs, ss, wald_xSq,
            wald_bins_joint, f_joint, f_wind, f_solar, f_load,
            x_sq_name, quadratic_headroom,
            bin_dummies, bin_edges, bin_active_names, bin_ref_idx, n_bins_eff,
            plain_logit, add_lags, fs_entity_fe, fs_month_fe,
            results_table=None, boot_df=None, boot_se=None,
            Z_cols_base=Z_cols_base, Z_sq_cols=Z_sq_cols,
        )

    # ------------------------------------------------------------------
    # 8. Clustered bootstrap
    # ------------------------------------------------------------------
    clusters   = np.array(list(entity_index.keys()))
    rng        = np.random.default_rng(seed)
    boot_coefs = []
    n_failed   = 0
    first_err  = None

    for _ in tqdm(range(n_boot), desc="Bootstrap"):
        sampled = rng.choice(clusters, size=len(clusters), replace=True)
        idx     = np.concatenate([entity_index[c] for c in sampled])

        y_b    = y_arr[idx]
        x_b    = x_arr[idx]
        W_b    = W_arr[idx]
        W_fs_b = W_fs_arr[idx] if W_fs_arr is not None else None
        Z_b    = Z_arr[idx]    if Z_arr is not None    else None

        try:
            coefs_b = _both_stages_boot(y_b, x_b, W_b, W_fs_b, Z_b)
            boot_coefs.append(coefs_b)
        except Exception as e:
            n_failed += 1
            if first_err is None:
                first_err = repr(e)

    if first_err:
        print(f"\n[Debug] First failure reason: {first_err}")

    boot_df    = pd.DataFrame(boot_coefs, columns=struct_names)
    boot_se    = boot_df.std().values
    boot_ci_lo = boot_df.quantile(0.025).values
    boot_ci_hi = boot_df.quantile(0.975).values

    results_table = pd.DataFrame({
        "coef"   : full_struct,
        "boot_se": boot_se,
        "ci_lo"  : boot_ci_lo,
        "ci_hi"  : boot_ci_hi,
    }, index=struct_names)
    results_table["z"] = (results_table["coef"] / results_table["boot_se"]).astype(float)
    results_table["p"] = 2 * (1 - scipy.stats.norm.cdf(
        np.abs(results_table["z"].values.astype(float))
    ))

    print(f"\nBootstrap: {len(boot_coefs)}/{n_boot} succeeded  "
          f"({n_failed} failed)\n")
    print(results_table.to_string(float_format="{:.4f}".format))

    if vhat_full is not None:
        print(f"\nvhat_full describe:\n{vhat_full.describe()}")
        print(f"vhat_full quantiles 1%/99%: {vhat_full.quantile([0.01, 0.99]).values}")

    return _build_return(
        all_data, W2_sm, all_Z_cols, fs, ss, wald_xSq,
        wald_bins_joint, f_joint, f_wind, f_solar, f_load,
        x_sq_name, quadratic_headroom,
        bin_dummies, bin_edges, bin_active_names, bin_ref_idx, n_bins_eff,
        plain_logit, add_lags, fs_entity_fe, fs_month_fe,
        results_table=results_table,
        boot_df=boot_df,
        boot_se=pd.Series(boot_se, index=struct_names),
        Z_cols_base=Z_cols_base, Z_sq_cols=Z_sq_cols,
    )


def _build_return(all_data, W2_sm, all_Z_cols, fs, ss, wald_xSq,
                  wald_bins_joint, f_joint, f_wind, f_solar, f_load,
                  x_sq_name, quadratic_headroom,
                  bin_dummies, bin_edges, bin_active_names, bin_ref_idx, n_bins_eff,
                  plain_logit, add_lags, fs_entity_fe, fs_month_fe,
                  *, results_table, boot_df, boot_se,
                  Z_cols_base, Z_sq_cols):
    """Helper to centralise return dict construction so we don't duplicate it.

    Keys for removed features (interaction, vhat*x, threshold, 2SLS, probit FS,
    size main effect, cubic instruments, cubic headroom) are kept with constant
    values so that
    downstream consumers (LaTeX tables, plots, cached pickles) remain compatible.
    """
    return {
        "all_data":             all_data,
        "W":                    W2_sm,
        "Z_cols":               all_Z_cols,
        "Z_cols_base":          Z_cols_base,
        "Z_sq_cols":            Z_sq_cols,
        "Z_cu_cols":            [],
        "first_stage":          fs,
        "second_stage":         ss,
        "wald_interaction":     None,
        "wald_quadratic":       wald_xSq,
        "wald_cubic":           None,
        "wald_bins_joint":      wald_bins_joint,
        "f_joint":              f_joint,
        "f_wind":               f_wind,
        "f_solar":              f_solar,
        "f_load":               f_load,
        "xS_name":              None,
        "x_sq_name":            x_sq_name if quadratic_headroom else None,
        "x_cu_name":            None,
        "vhatX_name":           None,
        "thresh_name":          None,
        "threshX_name":         None,
        "bin_dummies":          bin_dummies,
        "bin_edges":            bin_edges,
        "bin_names":            bin_active_names,
        "bin_ref_idx":          bin_ref_idx,
        "n_bins":               n_bins_eff,
        "include_vhat_x":       False,
        "include_interaction":  False,
        "quadratic_headroom":   quadratic_headroom,
        "cubic_headroom":       False,
        "include_threshold_x":  False,
        "binary_first_stage":   False,
        "linear_2sls":          False,
        "plain_logit":          plain_logit,
        "add_lags":             add_lags,
        "include_size_main":    False,
        "fs_entity_fe":         fs_entity_fe,
        "fs_month_fe":          fs_month_fe,
        "results_table":        results_table,
        "boot_df":              boot_df,
        "boot_se":              boot_se,
    }

In [ ]:
def leave_one_out_draws(x_col, eic_codes, *, n_boot=500, **kwargs):
    base = run_model(x_col, n_boot=n_boot, **kwargs)
    h_sample = base["all_data"][x_col].astype(float).to_numpy()  # full sample
    bundles = [_extract_draws(base, omitted="(none / full sample)")]
    for eic in eic_codes:
        try:
            out = run_model(x_col, n_boot=n_boot, leave_out=eic, **kwargs)
            bundles.append(_extract_draws(out, omitted=eic))
        except Exception as e:
            print(f"[skip] {eic}: {e!r}")
    return bundles, h_sample


def _extract_draws(out, omitted):
    """Keep structural coefs + bootstrap draws so ME(H) can be
    re-evaluated at any headroom value without refitting."""
    ss   = out["second_stage"]
    boot = out.get("boot_df")

    names = list(ss.params.index)
    def _is_lin(c):
        return ((c.startswith("im_") or c.startswith("ex_"))
                and not c.endswith("_sq") and not c.endswith("_cu")
                and not c.startswith("vhat") and not c.endswith("_x_size"))
    lin = [c for c in names if _is_lin(c)][0]
    sq  = out.get("x_sq_name")
    cu  = out.get("x_cu_name")

    bd = {
        "omitted": omitted,
        "b1": float(ss.params[lin]),
        "b2": float(ss.params[sq]) if sq else 0.0,
        "b3": float(ss.params[cu]) if cu else 0.0,
        "d1": None, "d2": None, "d3": None,
    }
    if boot is not None and lin in boot.columns:
        bd["d1"] = boot[lin].to_numpy().copy()
        bd["d2"] = (boot[sq].to_numpy() if (sq and sq in boot.columns)
                    else np.zeros_like(bd["d1"]))
        bd["d3"] = (boot[cu].to_numpy() if (cu and cu in boot.columns)
                    else np.zeros_like(bd["d1"]))
    return bd

def loo_table(bundles, h_eval):
    rows = []
    for bd in bundles:
        me = bd["b1"] + 2*bd["b2"]*h_eval + 3*bd["b3"]*h_eval**2
        row = {"omitted": bd["omitted"], "h_eval": h_eval, "coef": me,
               "ci_lo": np.nan, "ci_hi": np.nan, "boot_se": np.nan}
        if bd["d1"] is not None:
            md = bd["d1"] + 2*bd["d2"]*h_eval + 3*bd["d3"]*h_eval**2
            row["ci_lo"]   = float(np.quantile(md, 0.025))
            row["ci_hi"]   = float(np.quantile(md, 0.975))
            row["boot_se"] = float(np.std(md))
        rows.append(row)
    return pd.DataFrame(rows)

In [ ]:
def _panel(df, ax, title, color, eic_to_name, ylabel,
           eic_to_type=None, type_colors=None):
    base_mask = df["omitted"].str.contains("full sample", case=False, na=False)
    base = df[base_mask]
    pts = df[~base_mask].reset_index(drop=True)

    # Baseline reference band (full-sample coef +/- its bootstrap CI)
    if len(base):
        b = base.iloc[0]
        ax.axhspan(b["ci_lo"], b["ci_hi"], color="0.92", zorder=0,
                   label="Full-sample 95% CI")
        ax.axhline(b["coef"], color="0.45", lw=1.0, ls="--", zorder=1,
                   label="Full-sample estimate")

    y    = pts["coef"].values
    lo   = pts["coef"].values - pts["ci_lo"].values
    hi   = pts["ci_hi"].values - pts["coef"].values
    xpos = np.arange(len(pts))

    seen = []   # production types appearing in this panel (None == unknown)

    if eic_to_type is None:
        # ---- single-color mode (original behavior) ----
        ax.errorbar(xpos, y, yerr=[lo, hi], fmt="o", ms=4,
                    color=color, ecolor=color, elinewidth=1.0,
                    capsize=2.5, zorder=3)
    else:
        # ---- color-by-production-type mode ----
        type_list = [eic_to_type.get(e) for e in pts["omitted"]]
        for t in dict.fromkeys(type_list):                 # preserve appearance order
            idx = np.array([i for i, tt in enumerate(type_list) if tt == t])
            c = _UNKNOWN_COLOR if t is None else type_colors.get(t, "0.5")
            ax.errorbar(xpos[idx], y[idx], yerr=[lo[idx], hi[idx]],
                        fmt="o", ms=5, color=c, ecolor=c, elinewidth=1.0,
                        capsize=2.5, markeredgecolor="black",
                        markeredgewidth=0.4, zorder=3)
            seen.append(t)

    # x labels: plant names if a map is given, else shortened EIC
    labels = []
    for eic in pts["omitted"]:
        if eic_to_name and eic in eic_to_name:
            labels.append(eic_to_name[eic])
        else:
            labels.append(str(eic).replace("11WD", "").replace("11W", ""))
    ax.set_xticks(xpos)
    ax.set_xticklabels(labels, rotation=90, fontsize=7)

    ax.set_title(title, fontsize=13)
    ax.set_ylabel(ylabel)
    ax.set_xlabel("Omitted unit")
    ax.margins(x=0.02)
    from matplotlib.ticker import FormatStrFormatter
    ax.yaxis.set_major_formatter(FormatStrFormatter("%.2f"))
    return seen


def _ordinal(n):
    # 1->1st, 2->2nd, 3->3rd, 10->10th, etc.
    return f"{n}{'th' if 10 <= n % 100 <= 20 else {1:'st',2:'nd',3:'rd'}.get(n % 10, 'th')}"


def leave_one_out_plot(
    loo_neg,
    loo_pos,
    *,
    pct: float = 0.50,                 # evaluation percentile (0.10, 0.50, ...)
    eic_to_name: Optional[Dict[str, str]] = None,
    eic_to_type: Optional[Dict[str, str]] = None,   # EIC -> production_type
    type_colors: Optional[Dict[str, str]] = None,   # production_type -> color
    figsize=(12, 5.5),
    color_neg="#C99A4E",
    color_pos="#34699A",
    neg_title="Negative deviation",
    pos_title="Positive deviation",
):
    """Two-panel leave-one-out coefficient stability plot.

    If eic_to_type is given, points (and their CIs) are colored by production
    type and a type legend is shown; otherwise the per-direction colors
    (color_neg / color_pos) are used.
    """

    # accept a pandas Series as well as a dict
    if eic_to_type is not None and hasattr(eic_to_type, "to_dict"):
        eic_to_type = eic_to_type.to_dict()
    if eic_to_type is not None and type_colors is None:
        type_colors = DEFAULT_TYPE_COLORS

    p = int(round(pct * 100))
    ylabel = (f"Marginal effect at median headroom" if p == 50
              else f"Marginal effect at {_ordinal(p)} headroom percentile ")

    font_sizes = {
        "font.size": 14,
        "axes.titlesize": 16,
        "axes.labelsize": 12,
        "xtick.labelsize": 12,
        "ytick.labelsize": 12,
        "legend.fontsize": 13,
    }

    with plt.rc_context(font_sizes):
        fig, (axL, axR) = plt.subplots(1, 2, figsize=figsize)
        seen_neg = _panel(loo_neg, axL, neg_title, color_neg, eic_to_name, ylabel,
                          eic_to_type, type_colors)
        seen_pos = _panel(loo_pos, axR, pos_title, color_pos, eic_to_name, ylabel,
                          eic_to_type, type_colors)

    if eic_to_type is None:
        handles = [
            Line2D([0], [0], marker="o", color="0.3", lw=0, ms=5,
                   label="Leave-one-out estimate \u00b1 95% CI"),
            Line2D([0], [0], color="0.45", lw=1.0, ls="--",
                   label="Full-sample estimate"),
            Patch(facecolor="0.92", label="Full-sample 95% CI"),
        ]
    else:
        # union of types across both panels, in appearance order
        union = list(dict.fromkeys(list(seen_neg) + list(seen_pos)))
        handles = []
        for t in union:
            c = _UNKNOWN_COLOR if t is None else type_colors.get(t, "0.5")
            lbl = "Unknown" if t is None else t
            handles.append(Line2D([0], [0], marker="o", color=c, lw=0, ms=6,
                                  markeredgecolor="black", markeredgewidth=0.4,
                                  label=lbl))
        handles += [
            Line2D([0], [0], color="0.45", lw=1.0, ls="--",
                   label="Full-sample estimate"),
             Patch(facecolor="0.92", label="Full-sample 95% CI"),
        ]

    fig.legend(handles=handles, loc="lower center", ncol=min(len(handles), 4),
                frameon=False, bbox_to_anchor=(0.5, -0.04))
    fig.tight_layout()
    return fig


if __name__ == "__main__":
    pass

In [ ]:
def _structural_names(out: dict):
    src = list(out["second_stage"].params.index)
    x_sq = out.get("x_sq_name")
    x_cu = out.get("x_cu_name")
    drop = {"vhat", "const"}
    if x_sq:
        drop.add(x_sq)
    if x_cu:
        drop.add(x_cu)
    # structural linear term = the headroom column: starts with im_/ex_,
    # is not a squared/cubic/interaction/control, and not a fixed effect
    lin = [c for c in src
           if c not in drop
           and not c.startswith("vhat")
           and not c.endswith("_x_size")
           and not c.startswith("entity_")
           and not c.startswith("month_")
           and not c.endswith("_sq")
           and not c.endswith("_cu")
           and (c.startswith("im_") or c.startswith("ex_"))]
    if not lin:
        raise ValueError("Could not locate the headroom term in second_stage params.")
    return lin[0], x_sq, x_cu


def _coef(out, name):
    if name is None:
        return None
    return float(out["second_stage"].params[name])


def _curve(out, Hq, Href, center_at_median, ci):
    """Fitted f(H) curve + band on grid Hq, centered at Href.
    Uses bootstrap quantiles if boot_df is present, else analytic
    second-stage SEs via the delta method (valid for plain logit)."""
    x_lin, x_sq, x_cu = _structural_names(out)
    b1, b2, b3 = _coef(out, x_lin), _coef(out, x_sq), _coef(out, x_cu)

    def f(c1, c2, c3, H):
        y = c1 * H
        if c2 is not None:
            y = y + c2 * H ** 2
        if c3 is not None:
            y = y + c3 * H ** 3
        return y

    f_ref = f(b1, b2, b3, Href) if center_at_median else 0.0
    yhat = f(b1, b2, b3, Hq) - f_ref

    boot = out.get("boot_df")
    if boot is not None:
        # Bootstrap band (2SRI: propagates the generated-regressor uncertainty)
        B1 = boot[x_lin].values[:, None]
        cur = B1 * Hq[None, :]
        base = B1 * Href
        if x_sq:
            B2 = boot[x_sq].values[:, None]
            cur = cur + B2 * Hq[None, :] ** 2
            base = base + B2 * Href ** 2
        if x_cu:
            B3 = boot[x_cu].values[:, None]
            cur = cur + B3 * Hq[None, :] ** 3
            base = base + B3 * Href ** 3
        if center_at_median:
            cur = cur - base
        a = (1 - ci) / 2
        return yhat, np.quantile(cur, a, axis=0), np.quantile(cur, 1 - a, axis=0)

    # Analytic delta-method band (plain logit: no generated regressor)
    ss = out["second_stage"]
    names = [x_lin] + ([x_sq] if x_sq else []) + ([x_cu] if x_cu else [])
    Sigma = ss.cov_params().loc[names, names].values

    # gradient of (f(H) - f(Href)) w.r.t. [b1, (b2), (b3)]
    dH = Hq - (Href if center_at_median else 0.0)
    cols = [dH]
    if x_sq:
        cols.append(Hq ** 2 - (Href ** 2 if center_at_median else 0.0))
    if x_cu:
        cols.append(Hq ** 3 - (Href ** 3 if center_at_median else 0.0))
    G = np.column_stack(cols)                    # (n_grid, k)
    var = np.einsum("gi,ij,gj->g", G, Sigma, G)  # gᵀ Σ g per grid point
    se = np.sqrt(np.clip(var, 0, None))
    z = scipy.stats.norm.ppf(1 - (1 - ci) / 2)
    return yhat, yhat - z * se, yhat + z * se


def _dose_panel(out, ax, title, xlabel, pct_range, n_grid, ci, color,
                center_at_median, x_axis, out_plain=None, plain_color="0.5"):
    x_lin, _, _ = _structural_names(out)
    xv = out["all_data"][x_lin].astype(float).values
    q = np.linspace(pct_range[0], pct_range[1], n_grid)
    Hq = np.percentile(xv, q)
    Href = np.percentile(xv, 50)

    xplot = Hq if x_axis == "gw" else q
    xref = Href if x_axis == "gw" else 50.0

    yhat, lo, hi = _curve(out, Hq, Href, center_at_median, ci)

    if center_at_median:
        ax.axhline(0, color="k", lw=0.9, ls=(0, (4, 3)), zorder=1)
        ax.axvline(xref, color="0.55", lw=0.8, ls=":", zorder=1)

    # Plain-logit overlay (drawn first, underneath the 2SRI curve)
    if out_plain is not None:
        yhp, lop, hip = _curve(out_plain, Hq, Href, center_at_median, ci)
        if lop is not None:
            ax.fill_between(xplot, lop, hip, facecolor="0.55",
                            edgecolor="none", alpha=0.45, zorder=1.5)
        ax.plot(xplot, yhp, color=plain_color, lw=2.0, ls=":", zorder=2.5)


    # 2SRI curve
    if lo is not None:
        ax.fill_between(xplot, lo, hi, color=color, alpha=0.25, lw=0, zorder=2)
    ax.plot(xplot, yhat, color=color, lw=2.2, zorder=3)

    ax.set_title(title)
    ax.set_xlabel(xlabel)
    if x_axis == "percentile":
        ax.set_xticks(list(range(10, 101, 10)))
        ax.set_xticklabels([f"$p_{{{t}}}$" for t in range(10, 101, 10)])
    ax.margins(x=0)


def dose_response_plot(
    out_neg_sq: dict,
    out_pos_sq: dict,
    *,
    overlay_plain: bool = False,
    out_neg_plain: Optional[dict] = None,
    out_pos_plain: Optional[dict] = None,
    plain_color: str = "0.5",
    x_axis: str = "percentile",
    center_at_median: bool = False,
    pct_range: Tuple[float, float] = (1, 100),
    n_grid: int = 200,
    ci: float = 0.95,
    color_neg: str = "#C99A4E",
    color_pos: str = "#34699A",
    figsize: Tuple[float, float] = (11, 4.5),
    neg_title: str = "Negative deviation",
    pos_title: str = "Positive deviation",
    neg_xlabel: Optional[str] = None,
    pos_xlabel: Optional[str] = None,
):
    """
    Two-panel continuous dose-response plot (import/neg | export/pos).

    overlay_plain : if True, overlay the plain-logit (uncorrected) dose-response
        as a dashed grey line+band. Requires out_neg_plain and out_pos_plain
        (run_model outputs with plain_logit=True, same functional form).
    x_axis : "percentile" (default) plots against the headroom percentile
        (decile ticks). "gw" plots against headroom in GW (one x-unit = 1 GW).
    center_at_median : if True, subtract f(H_p50) so the curve reads relative
        to the median hour (reference line at p50). If False (default), plot
        the uncentered contribution f(H)=b1*H+b2*H^2.
    """
    if x_axis not in ("percentile", "gw"):
        raise ValueError("x_axis must be 'percentile' or 'gw'.")
    if overlay_plain and (out_neg_plain is None or out_pos_plain is None):
        raise ValueError(
            "overlay_plain=True requires out_neg_plain and out_pos_plain."
        )

    if neg_xlabel is None:
        neg_xlabel = ("Import headroom (GW)" if x_axis == "gw"
                      else "Import headroom decile")
    if pos_xlabel is None:
        pos_xlabel = ("Export headroom (GW)" if x_axis == "gw"
                      else "Export headroom decile")

    fig, (axL, axR) = plt.subplots(1, 2, figsize=figsize, sharey=True)
    _dose_panel(out_neg_sq, axL, neg_title, neg_xlabel,
                pct_range, n_grid, ci, color_neg, center_at_median, x_axis,
                out_plain=out_neg_plain if overlay_plain else None,
                plain_color=plain_color)
    _dose_panel(out_pos_sq, axR, pos_title, pos_xlabel,
                pct_range, n_grid, ci, color_pos, center_at_median, x_axis,
                out_plain=out_pos_plain if overlay_plain else None,
                plain_color=plain_color)

    if center_at_median:
        ylabel = r"Log-odds vs. median headroom"
    else:
        ylabel = r"Log-odds contribution, $f(H)$"
    axL.set_ylabel(ylabel)

    # Neutral-grey legend: solid = 2SRI, dashed = plain logit, patch = CI.
    grey = "0.4"
    legend_handles = [
        Line2D([0], [0], color=grey, lw=2.2, label="2SRI coefficient"),
    ]
    if overlay_plain:
        legend_handles.append(
            Line2D([0], [0], color=grey, lw=2.0, ls=":",
                   label="Plain logit coefficient")
        )
    legend_handles.append(
        Patch(facecolor=grey, alpha=0.25, label="95% CI")
    )
    if center_at_median:
        legend_handles.append(
            Line2D([0], [0], color="0.55", lw=0.8, ls=":",
                   label="Median headroom")
        )

    fig.legend(handles=legend_handles, loc="lower center",
               ncol=len(legend_handles), frameon=False,
               bbox_to_anchor=(0.5, -0.04))
    fig.tight_layout()
    return fig


if __name__ == "__main__":
    pass

In [ ]:
def plot_bin_marginal_effects(
    out_neg_bins,
    out_pos_bins,
    overlay_plain: bool = False,
    out_neg_plain=None,
    out_pos_plain=None,
    plain_color: str = "#888780",
    figsize=(11, 4.5),
    bar_color_neg="#C99A4E",
    bar_color_pos="#34699A",
    ref_color="#888780",
    use_bootstrap_ci: bool = True,
    confidence: float = 0.95,
    title_neg: str = "Negative deviation",
    title_pos: str = "Positive deviation",
    ylabel: str = "Log-odds vs. reference decile",
    save_path: str = None,
    dpi: int = 150,
):
    """
    Plot bin-dummy marginal effects side-by-side for withholding and push-in.

    Parameters
    ----------
    out_neg_bins / out_pos_bins : dict
        run_model() output with bin_dummies=True. Used as the baseline (bars).
    overlay_plain : bool
        If True, overlay plain-logit bin coefficients as a dashed line+markers.
    out_neg_plain / out_pos_plain : dict, optional
        run_model() output with bin_dummies=True and plain_logit=True.
        Required if overlay_plain=True.
    plain_color : str
        Color for the overlaid plain-logit line and markers.
    bar_color : str
        Color for the baseline (2SRI) bars.
    ref_color : str
        Color for the reference bin bar.
    confidence : float
        CI level (default 0.95).
    other args : standard matplotlib plumbing.
    """
    if overlay_plain and (out_neg_plain is None or out_pos_plain is None):
        raise ValueError(
            "overlay_plain=True requires out_neg_plain and out_pos_plain."
        )

    def _extract(out: dict, n_bins_default: int = 10):
        bin_names  = out.get("bin_names", [])
        n_bins_eff = out.get("n_bins", n_bins_default)

        if not bin_names:
            raise ValueError("No bin_names found. Did you run with bin_dummies=True?")

        def _bin_num(nm):
            m = re.search(r'_bin(\d+)_', nm)
            return int(m.group(1)) if m else None

        active_nums = [_bin_num(nm) for nm in bin_names]
        all_nums    = list(range(1, n_bins_eff + 1))
        missing     = sorted(set(all_nums) - set(active_nums))
        ref_num_1   = missing[0] if missing else 1

        step = 100 // n_bins_eff
        def _pct_label(k1):
            upper = k1 * step
            return f"$p_{{{upper}}}$"

        labels  = [_pct_label(k) for k in range(1, n_bins_eff + 1)]
        ref_idx = ref_num_1 - 1

        ss      = out["second_stage"]
        rt      = out.get("results_table")
        boot_df = out.get("boot_df")

        coefs = np.zeros(n_bins_eff)
        ci_lo = np.zeros(n_bins_eff)
        ci_hi = np.zeros(n_bins_eff)

        try:
            from scipy.stats import norm as _norm
            z_crit = _norm.ppf(1 - (1 - confidence) / 2)
        except ImportError:
            z_crit = 1.96

        for nm in bin_names:
            k1  = _bin_num(nm)
            idx = k1 - 1
            coef = float(ss.params[nm])
            coefs[idx] = coef

            if use_bootstrap_ci and boot_df is not None and nm in boot_df.columns:
                draws      = boot_df[nm].dropna().values
                alpha      = 1 - confidence
                ci_lo[idx] = np.percentile(draws, 100 * alpha / 2)
                ci_hi[idx] = np.percentile(draws, 100 * (1 - alpha / 2))
            elif rt is not None and nm in rt.index:
                se = float(rt.loc[nm, "boot_se"])
                ci_lo[idx] = coef - z_crit * se
                ci_hi[idx] = coef + z_crit * se
            else:
                se = float(ss.bse[nm])
                ci_lo[idx] = coef - z_crit * se
                ci_hi[idx] = coef + z_crit * se

        return labels, coefs, ci_lo, ci_hi, ref_idx, n_bins_eff

    # Baseline bins
    labels_neg, coefs_neg, lo_neg, hi_neg, ref_neg, _ = _extract(out_neg_bins)
    labels_pos, coefs_pos, lo_pos, hi_pos, ref_pos, _ = _extract(out_pos_bins)

    # Plain logit overlay
    if overlay_plain:
        _, plain_neg, _, _, _, _ = _extract(out_neg_plain)
        _, plain_pos, _, _, _, _ = _extract(out_pos_plain)

    fig, axes = plt.subplots(1, 2, figsize=figsize)

    panel_data = [
        (axes[0], labels_neg, coefs_neg, lo_neg, hi_neg, ref_neg, title_neg,
         out_neg_plain if overlay_plain else None, bar_color_neg,
         "Import headroom decile"),
        (axes[1], labels_pos, coefs_pos, lo_pos, hi_pos, ref_pos, title_pos,
         out_pos_plain if overlay_plain else None, bar_color_pos,
         "Export headroom decile"),
    ]

    for ax, labels, coefs, lo, hi, ref_idx, title, plain_out, panel_bar_color, panel_xlabel in panel_data:
        xs = np.arange(len(labels))

        if overlay_plain:
            # Grouped bars: 2SRI on left, plain logit on right
            bar_w = 0.36
            offset = bar_w / 2 + 0.02

            # Extract plain logit results
            _, plain_coefs, plain_lo, plain_hi, _, _ = _extract(plain_out)

            # 2SRI bars (left)
            sri_colors = [ref_color if i == ref_idx else panel_bar_color
                          for i in range(len(coefs))]
            ax.bar(xs - offset, coefs, width=bar_w,
                   color=sri_colors, zorder=3, label="_nolegend_")
            ax.errorbar(xs - offset, coefs,
                        yerr=[coefs - lo, hi - coefs],
                        fmt="none", ecolor="black",
                        elinewidth=1.0, capsize=2.5, capthick=1.0, zorder=4)

            # Plain logit bars (right)
            plain_colors = [ref_color if i == ref_idx else plain_color
                            for i in range(len(plain_coefs))]
            ax.bar(xs + offset, plain_coefs, width=bar_w,
                   color=plain_colors, zorder=3, label="_nolegend_")
            ax.errorbar(xs + offset, plain_coefs,
                        yerr=[plain_coefs - plain_lo, plain_hi - plain_coefs],
                        fmt="none", ecolor="black",
                        elinewidth=1.0, capsize=2.5, capthick=1.0, zorder=4)

        else:
            sri_colors = [ref_color if i == ref_idx else panel_bar_color
                          for i in range(len(coefs))]
            ax.bar(xs, coefs, width=0.6, color=sri_colors, zorder=3)
            ax.errorbar(xs, coefs,
                        yerr=[coefs - lo, hi - coefs],
                        fmt="none", ecolor="black",
                        elinewidth=1.0, capsize=3, capthick=1.0, zorder=4)

        ax.axhline(0, color="black", linewidth=0.8, linestyle="--", zorder=2)
        ax.text(ref_idx, 0.01, "ref", ha="center", va="bottom",
                fontsize=8, color=ref_color)

        ax.set_xticks(xs)
        ax.set_xticklabels(labels)
        ax.set_xlabel(panel_xlabel)
        ax.set_ylabel(ylabel)
        ax.set_title(title)
    
    # Sync y-limits: use the wider of the two panels for both
    ylims = [ax.get_ylim() for ax in axes]
    y_min = min(lim[0] for lim in ylims)
    y_max = max(lim[1] for lim in ylims)
    for ax in axes:
        ax.set_ylim(y_min, y_max)

    class _ErrorBarHandle:
        """Sentinel — picks up styling from the matching legend handler."""
        pass

    class _ErrorBarHandler(HandlerBase):
        """Draws an I-beam (vertical line with caps top and bottom)."""
        def create_artists(self, legend, orig_handle, xdescent, ydescent,
                           width, height, fontsize, trans):
            x_mid = xdescent + width / 2
            y_top = ydescent + height
            y_bot = ydescent
            cap_half = width * 0.18
            stem = Line2D([x_mid, x_mid], [y_bot, y_top],
                          color="black", linewidth=1.0, transform=trans)
            top_cap = Line2D([x_mid - cap_half, x_mid + cap_half],
                             [y_top, y_top],
                             color="black", linewidth=1.0, transform=trans)
            bot_cap = Line2D([x_mid - cap_half, x_mid + cap_half],
                             [y_bot, y_bot],
                             color="black", linewidth=1.0, transform=trans)
            return [stem, top_cap, bot_cap]

    grey = "0.5"
    legend_handles = [
        mpatches.Patch(color=grey, label="2SRI coefficient"),
    ]
    if overlay_plain:
        legend_handles.append(
            mpatches.Patch(color=grey, hatch="///", label="Plain logit coefficient")
        )
    legend_handles.append(
        (_ErrorBarHandle(), f"{int(confidence*100)}% CI")
    )

    fig.legend(
        handles=[h if not isinstance(h, tuple) else h[0] for h in legend_handles],
        labels=[h.get_label() if not isinstance(h, tuple) else h[1]
                for h in legend_handles],
        handler_map={_ErrorBarHandle: _ErrorBarHandler()},
        loc="lower center",
        ncol=len(legend_handles),
        bbox_to_anchor=(0.5, -0.04),
        frameon=False
    )

    plt.tight_layout()

    if save_path:
        fig.savefig(save_path, dpi=dpi, bbox_inches="tight")
        print(f"Saved → {save_path}")
    
    return fig, axes

In [ ]:
def short_label_raw(ptype):
    return ptype.replace('Fossil ', '')

def get_entity_raw_probs(out):
    """Empirical per-entity rate of the binary outcome, on the estimation sample."""
    ss = out['second_stage']
    all_data = out['all_data'].copy()
    # endog is the 0/1 outcome; align by the rows actually used in the fit
    all_data['_y'] = pd.Series(np.asarray(ss.model.endog),
                               index=ss.model.data.row_labels,
                               name='_y')
    raw = all_data.groupby('_entity_id')['_y'].mean()
    raw.index = raw.index.str.replace('entity_', '', regex=False)
    return raw

In [ ]:
def save_result(res: dict, name: str):
    """Save a run_model result dict to disk."""
    path = RESULTS_DIR / f"{name}.pkl"
    with open(path, "wb") as f:
        pickle.dump(res, f, protocol=pickle.HIGHEST_PROTOCOL)
    print(f"Saved → {path}")

In [ ]:
def load_result(name: str) -> dict:
    """Load a previously saved run_model result dict."""
    path = RESULTS_DIR / f"{name}.pkl"
    with open(path, "rb") as f:
        res = pickle.load(f)
    print(f"Loaded ← {path}")
    return res

In [ ]:
def _stars(p: float) -> str:
    if p < 0.001: return "***"
    if p < 0.01:  return "**"
    if p < 0.05:  return "*"
    return ""

def _fmt_num_plain(x: Optional[float], ndp: int = 2, tiny_to_zero: bool = False) -> str:
    if x is None:
        return ""
    xf = float(x)
    if math.isnan(xf) or math.isinf(xf):
        return ""
    if tiny_to_zero and abs(xf) < 0.005:
        return "0.00"
    # no scientific notation: clamp formatting
    return f"{xf:.{ndp}f}"

def _coef_se_p(res, name: str):
    if res is None:
        return None, None, None
    try:
        b = float(res.params[name])
    except Exception:
        return None, None, None
    se = None
    p = None
    try:
        se = float(res.bse[name])
    except Exception:
        pass
    try:
        p = float(res.pvalues[name])
    except Exception:
        pass
    return b, se, p

def _joint_f(fs_res, restriction: str, include_p: bool = True) -> str:
    if fs_res is None:
        return ""
    try:
        t = fs_res.f_test(restriction)
        fval = float(np.asarray(t.fvalue).squeeze())
        pval = float(np.asarray(t.pvalue).squeeze())
        if include_p:
            # forces tiny p-values to render "0.00" after rounding
            return f"{fval:.2f} (p={pval:.2f})"
        return f"{fval:.2f}"
    except Exception:
        return ""

In [ ]:
_DEFAULT_COLS = [
    "im_headroom",
    "ex_headroom",
    "gas",
    "carbon",
    "coal",
    "wind",
    "solar",
    "load",
]

_DEFAULT_LABELS = {
    "im_headroom":                          r"Import headroom",
    "ex_headroom":                          r"Export headroom",
    "gas":                                  r"Gas price",
    "carbon":                               r"Carbon price",
    "coal":                                 r"Coal price",
    "wind":                                 r"Wind",
    "solar":                                r"Solar",
    "load":                                 r"Load",
}

_DEFAULT_UNITS = {
    "im_headroom":                          "GW",
    "ex_headroom":                          "GW",
    "wind":                                 "GW",
    "solar":                                "GW",
    "gas":                                  "EUR/MWh",
    "carbon":                               "EUR/t",
    "coal":                                 "USD/t",
    "load":                                 "GW"

}


def make_summary_stats_latex(
    df: pd.DataFrame,
    df_neg: pd.DataFrame = None,
    df_pos: pd.DataFrame = None,
    columns=None,
    labels=None,
    units=None,
    caption: str = "Summary statistics",
    label: str = "tab:summary",
    decimals: int = 2,
    use_thousands_sep: bool = False,
) -> str:
    """
    LaTeX summary table with Min, P25, P50, P75, Mean, Std. Dev., Max.

    Source frames:
      im_headroom  — computed from df_neg (falls back to df if not given)
      ex_headroom  — computed from df_pos (falls back to df if not given)
      all others   — computed from df

    Defaults:
      columns  — im_headroom, ex_headroom, prices, wind, solar, load
      labels   — human-readable names for the default columns
      units    — units for the default columns (wind/solar converted MW -> GW)
    """
    if columns is None:
        columns = _DEFAULT_COLS
    if labels is None:
        labels = _DEFAULT_LABELS.copy()
    else:
        # merge: caller overrides take precedence
        tmp = _DEFAULT_LABELS.copy()
        tmp.update(labels)
        labels = tmp
    if units is None:
        units = _DEFAULT_UNITS.copy()
    else:
        tmp = _DEFAULT_UNITS.copy()
        tmp.update(units)
        units = tmp

    # Per-column source frame. im_headroom -> df_neg, ex_headroom -> df_pos,
    # everything else -> df. Missing neg/pos frames fall back to df.
    col_source = {
        "im_headroom": df_neg if df_neg is not None else df,
        "ex_headroom": df_pos if df_pos is not None else df,
    }

    zero_eps = 0.5 * 10 ** (-decimals)

    def fmt(x):
        if x is None:
            return ""
        xf = float(x)
        if math.isnan(xf) or math.isinf(xf):
            return ""
        if abs(xf) < zero_eps:
            xf = 0.0
        if use_thousands_sep:
            return f"{xf:,.{decimals}f}"
        return f"{xf:.{decimals}f}"

    lines = [
        r"\begin{table}[htbp]",
        r"\centering",
        rf"\caption{{{caption}}}",
        rf"\label{{{label}}}",
        r"\begin{tabular}{lccccccc}",
        r"\hline",
        r" & Min & P25 & P50 & P75 & Mean & Std.\ Dev. & Max \\",
        r"\hline",
    ]

    for col in columns:
        src = col_source.get(col, df)
        if col not in src.columns:
            continue

        s = pd.to_numeric(src[col], errors="coerce").dropna()
        if s.empty:
            mn = p25 = p50 = p75 = mu = sd = mx = None
        else:
            mn  = s.min()
            p25 = s.quantile(0.25)
            p50 = s.quantile(0.50)
            p75 = s.quantile(0.75)
            mu  = s.mean()
            sd  = s.std(ddof=1)
            mx  = s.max()

        row_label = labels.get(col, col)
        if col in units and units[col]:
            row_label = f"{row_label} [{units[col]}]"

        lines.append(
            f"{row_label} & {fmt(mn)} & {fmt(p25)} & {fmt(p50)} & {fmt(p75)} & "
            f"{fmt(mu)} & {fmt(sd)} & {fmt(mx)} \\\\"
        )

    lines += [r"\hline", r"\end{tabular}", r"\end{table}"]
    return "\n".join(lines)

In [ ]:
def make_iv_2sri_latex_table(
    out_neg,
    out_pos,
    first_stage_joint_restriction: str = "wind_sl = 0, solar_sl = 0, load_sl = 0",
    caption: str = "2SRI regression results",
    label: str = "tab:iv_2sri",
    ndp: int = 2,
    ndp_fs_r2: int = 3,
    include_joint_p: bool = False,
    include_first_stage: bool = True,
    include_vhat: bool = True,
    appendix_table: bool = False,
    slides: bool = False,
    varmap: Optional[Dict[str, List[str]]] = None,
) -> str:
    """
    2-column LaTeX table (Negative deviation | Positive deviation).

    Auto-detects per column:
      - treatment type: headroom (continuous), congestion (binary), or bin dummies
      - whether a squared term is present
      - whether bin dummies are used (bin_dummies=True in run_model output)
      - whether a first stage exists (2SRI logit) or not (plain logit)

    When both columns use bin dummies with the same n_bins, bin rows are
    printed side-by-side using shared percentile labels H_t^{p10}, H_t^{p20},
    etc. — no GW ranges, compact layout.

    Modes (mutually exclusive):
      slides=False, appendix_table=False (default):
          Structural params only, controls suppressed with footnote,
          SEs on separate line below coefficient.
      appendix_table=True:
          Full coefficient table in footnotesize, SEs on same line in brackets.
      slides=True:
          Compact scriptsize table for Beamer. Structural params only,
          SEs inline. No table/caption/threeparttable wrapper.
    """
    if slides and appendix_table:
        raise ValueError("slides=True and appendix_table=True are mutually exclusive.")

    # ------------------------------------------------------------------
    # Internal helpers
    # ------------------------------------------------------------------
    def _find_param(res, candidates):
        for c in candidates:
            if c in res.params.index:
                return c
        return None

    def _coef_se_p(res, key):
        if key is None or key not in res.params.index:
            return None, None, None
        return float(res.params[key]), float(res.bse[key]), float(res.pvalues[key])

    def _stars(p):
        if p is None:
            return ""
        if p < 0.01:
            return "***"
        if p < 0.05:
            return "**"
        if p < 0.1:
            return "*"
        return ""

    def _fmt_num(x, dp):
        if x is None:
            return ""
        return f"{x:.{dp}f}"

    def _get_attr(obj, attr, default=None):
        return getattr(obj, attr, default)

    def _joint_f_stat(fs, restriction, include_p=False):
        if fs is None:
            return ""
        try:
            result = fs.f_test(restriction)
            stat = float(np.asarray(result.fvalue).ravel()[0])
            pval = float(np.asarray(result.pvalue).ravel()[0])
            s = f"{stat:.1f}"
            if include_p:
                s += f" (p={pval:.3f})"
            return s
        except Exception:
            return ""

    def _parse_bin_label(bin_name: str, n_bins: int = 10) -> str:
        """
        Convert e.g. 'im_headroom_bin3_[4.91,6.99]'
        → '$H_{t}^{p30}$'  (for n_bins=10)

        Uses upper-percentile decile labelling so im_ and ex_ bins
        with the same index share the same label and can be printed
        side-by-side in one row.
        """
        m = re.search(r'_bin(\d+)_', bin_name)
        if not m:
            return bin_name
        k = int(m.group(1))         # 1-indexed
        step = 100 // n_bins
        upper = k * step
        return rf"$H_{{t}}^{{p{upper}}}$"


    # ------------------------------------------------------------------
    # Wrapper: merges bootstrap SEs and p-values into analytic vectors
    # ------------------------------------------------------------------
    class _WrappedResults:
        def __init__(self, out: dict):
            ss = out["second_stage"]
            self.params    = ss.params
            self.nobs      = ss.nobs
            self.prsquared = getattr(ss, "prsquared", None)
            self.is_plain  = bool(out.get("plain_logit", False))

            bse = ss.bse.copy()
            boot_se = out.get("boot_se")
            if boot_se is not None:
                for param_name in boot_se.index:
                    if param_name in bse.index:
                        bse[param_name] = boot_se[param_name]
            self.bse = bse

            pvalues = ss.pvalues.copy()
            rt = out.get("results_table")
            if rt is not None:
                for param_name in rt.index:
                    if param_name in pvalues.index:
                        pvalues[param_name] = rt.loc[param_name, "p"]
            self.pvalues = pvalues

    res_pos = _WrappedResults(out_pos)
    res_neg = _WrappedResults(out_neg)
    fs_pos  = out_pos.get("first_stage")
    fs_neg  = out_neg.get("first_stage")

    # ------------------------------------------------------------------
    # Model-type flags (per column)
    # ------------------------------------------------------------------
    pos_has_fs    = (fs_pos is not None) and not res_pos.is_plain
    neg_has_fs    = (fs_neg is not None) and not res_neg.is_plain
    has_bins_pos  = bool(out_pos.get("bin_dummies", False))
    has_bins_neg  = bool(out_neg.get("bin_dummies", False))
    bin_names_pos = out_pos.get("bin_names", []) if has_bins_pos else []
    bin_names_neg = out_neg.get("bin_names", []) if has_bins_neg else []
    n_bins_pos    = out_pos.get("n_bins", len(bin_names_pos) + 1) if has_bins_pos else 10
    n_bins_neg    = out_neg.get("n_bins", len(bin_names_neg) + 1) if has_bins_neg else 10

    # Side-by-side layout only when both columns use same n_bins
    both_bins_same_n = (has_bins_pos and has_bins_neg
                        and n_bins_pos == n_bins_neg)

    # ------------------------------------------------------------------
    # Auto-detection helpers (non-bin columns)
    # ------------------------------------------------------------------
    def _detect_treatment(res, prefix: str):
        try:
            params = res.params.index
        except Exception:
            return None
        if f"{prefix}_congestion" in params:
            return "congestion"
        if f"{prefix}_headroom" in params or f"{prefix}_tightness" in params:
            return "headroom"
        return None

    def _has_quadratic(out: dict) -> bool:
        if out.get("bin_dummies"):
            return False
        if "quadratic_headroom" in out:
            return bool(out.get("quadratic_headroom"))
        x_sq_name = out.get("x_sq_name")
        if x_sq_name is None:
            return False
        try:
            return x_sq_name in out["second_stage"].params.index
        except Exception:
            return False

    treat_pos  = None if has_bins_pos else _detect_treatment(res_pos, "ex")
    treat_neg  = None if has_bins_neg else _detect_treatment(res_neg, "im")
    has_sq_pos = _has_quadratic(out_pos)
    has_sq_neg = _has_quadratic(out_neg)

    # ------------------------------------------------------------------
    # Bin label maps: percentile label → raw param name
    # ------------------------------------------------------------------
    bin_label_map_pos = {}
    if has_bins_pos:
        for nm in bin_names_pos:
            lbl = _parse_bin_label(nm, n_bins_pos)
            bin_label_map_pos[lbl] = nm

    bin_label_map_neg = {}
    if has_bins_neg:
        for nm in bin_names_neg:
            lbl = _parse_bin_label(nm, n_bins_neg)
            bin_label_map_neg[lbl] = nm

    # ------------------------------------------------------------------
    # Variable map (non-bin structural params)
    # ------------------------------------------------------------------
    default_varmap = {
        "Intercept": ["const", "Intercept"],
        r"$H^{ex}_{t}$": [
            "ex_headroom", "ex_tightness", "export_headroom",
            "ex_headroom_gw", "ex_tightness_gw",
        ],
        r"$(H^{ex}_{t})^{2}$": ["ex_headroom_sq", "ex_tightness_sq"],
        r"$H^{im}_{t}$": [
            "im_headroom", "im_tightness", "import_headroom",
            "im_headroom_gw", "im_tightness_gw",
        ],
        r"$(H^{im}_{t})^{2}$": ["im_headroom_sq", "im_tightness_sq"],
        r"$C^{ex}_{t}$": ["ex_congestion", "export_congestion"],
        r"$C^{im}_{t}$": ["im_congestion", "import_congestion"],
        r"$\hat{v}_{it}$": ["vhat", "v_hat", "uhat", "resid"],
        r"$wind_{t}$":          ["wind", "Wind_Onshore", "Wind"],
        r"$solar_{t}$":         ["solar"],
        r"$load_{t}$":          ["load"],
        r"$wind_{t-1}$":        ["L1_wind"],
        r"$solar_{t-1}$":       ["L1_solar"],
        r"$load_{t-1}$":        ["L1_load"],
        r"$wind_{t-2}$":        ["L2_wind"],
        r"$solar_{t-2}$":       ["L2_solar"],
        r"$load_{t-2}$":        ["L2_load"],
        r"$wind^{nb}_{t-1}$":   ["L1_wind_sl"],
        r"$solar^{nb}_{t-1}$":  ["L1_solar_sl"],
        r"$load^{nb}_{t-1}$":   ["L1_load_sl"],
        r"$wind^{nb}_{t-2}$":   ["L2_wind_sl"],
        r"$solar^{nb}_{t-2}$":  ["L2_solar_sl"],
        r"$load^{nb}_{t-2}$":   ["L2_load_sl"],
        r"$carbon_{t}$":        ["carbon"],
        r"$gas_{t}$":           ["gas"],
        r"$coal_{t}$":          ["coal"],
    }

    if varmap is None:
        varmap = default_varmap
    else:
        tmp = default_varmap.copy()
        tmp.update(varmap)
        varmap = tmp

    if not include_vhat:
        varmap = {k: v for k, v in varmap.items() if k != r"$\hat{v}_{it}$"}

    keys_pos = {lbl: _find_param(res_pos, cands) for lbl, cands in varmap.items()}
    keys_neg = {lbl: _find_param(res_neg, cands) for lbl, cands in varmap.items()}

    # ------------------------------------------------------------------
    # Row order
    # ------------------------------------------------------------------
    structural_rows = ["Intercept"]

    if both_bins_same_n:
        # Side-by-side: shared percentile labels, de-duplicated, in bin order
        shared_bin_labels = list(dict.fromkeys(
            list(bin_label_map_neg.keys()) +
            list(bin_label_map_pos.keys())
        ))
        structural_rows += shared_bin_labels

    else:
        # Interleave ex and im treatment rows; column suppression handled
        # via ex_only_labels / im_only_labels at render time

        # ex (pos) treatment
        if treat_pos == "congestion":
            structural_rows.append(r"$C^{ex}_{t}$")
        elif treat_pos == "headroom":
            structural_rows.append(r"$H^{ex}_{t}$")
            if has_sq_pos:
                structural_rows.append(r"$(H^{ex}_{t})^{2}$")

        # im (neg) treatment
        if treat_neg == "congestion":
            structural_rows.append(r"$C^{im}_{t}$")
        elif treat_neg == "headroom":
            structural_rows.append(r"$H^{im}_{t}$")
            if has_sq_neg:
                structural_rows.append(r"$(H^{im}_{t})^{2}$")

    # vhat row
    if include_vhat:
        vhat_lbl = r"$\hat{v}_{it}$"
        pos_vhat_key = (keys_pos.get(vhat_lbl) if not has_bins_pos
                        else _find_param(res_pos, ["vhat", "v_hat"]))
        neg_vhat_key = (keys_neg.get(vhat_lbl) if not has_bins_neg
                        else _find_param(res_neg, ["vhat", "v_hat"]))
        if pos_vhat_key is not None or neg_vhat_key is not None:
            structural_rows.append(vhat_lbl)

    control_rows = [
        r"$wind_{t}$", r"$solar_{t}$", r"$load_{t}$",
        r"$wind_{t-1}$", r"$solar_{t-1}$", r"$load_{t-1}$",
        r"$wind_{t-2}$", r"$solar_{t-2}$", r"$load_{t-2}$",
        r"$wind^{nb}_{t-1}$", r"$solar^{nb}_{t-1}$", r"$load^{nb}_{t-1}$",
        r"$wind^{nb}_{t-2}$", r"$solar^{nb}_{t-2}$", r"$load^{nb}_{t-2}$",
        r"$carbon_{t}$", r"$gas_{t}$", r"$coal_{t}$",
    ]
    control_rows = [r for r in control_rows
                    if keys_pos.get(r) is not None or keys_neg.get(r) is not None]

    coef_rows = structural_rows + (control_rows if appendix_table else [])

    # Labels that belong exclusively to one column (non-bin structural)
    ex_only_labels = {
        r"$H^{ex}_{t}$", r"$(H^{ex}_{t})^{2}$", r"$C^{ex}_{t}$",
    }
    im_only_labels = {
        r"$H^{im}_{t}$", r"$(H^{im}_{t})^{2}$", r"$C^{im}_{t}$",
    }

    # ------------------------------------------------------------------
    # Footnote strings
    # ------------------------------------------------------------------
    using_boot = (out_pos.get("boot_se") is not None
                  or out_neg.get("boot_se") is not None)
    se_note_text = (
        "Clustered bootstrap standard errors in parentheses"
        if using_boot
        else "Clustered standard errors in parentheses"
    )

    same_line_se = slides or appendix_table

    # ------------------------------------------------------------------
    # Header
    # ------------------------------------------------------------------
    lines: List[str] = []

    if slides:
        lines += [
            r"\centering",
            r"\scriptsize",
            r"\setlength{\tabcolsep}{3pt}",
            r"\renewcommand{\arraystretch}{1.0}",
            r"\begin{adjustbox}{max width=\textwidth}",
            r"\begin{tabular}{lcc}",
            r"\toprule",
            r" & Negative deviation & Positive deviation \\",
            r"\midrule",
        ]
    else:
        lines += [
            r"\begin{table}[]",
            r"\centering",
            r"\begin{adjustbox}{max width=\textwidth}",
            r"\begin{threeparttable}",
            rf"\caption{{{caption}}}",
            rf"\label{{{label}}}",
        ]
        lines += [
            r"\begin{tabular}{lcc}",
            r"\toprule",
            r" & Negative deviation & Positive deviation \\",
            r"\midrule",
        ]

    # ------------------------------------------------------------------
    # Coefficient rows
    # ------------------------------------------------------------------
    for lbl in coef_rows:

        # Determine whether this label is a bin label for each column
        is_bin_pos = lbl in bin_label_map_pos
        is_bin_neg = lbl in bin_label_map_neg

        # Resolve param key
        if is_bin_pos:
            pos_key = bin_label_map_pos[lbl]
        else:
            pos_key = keys_pos.get(lbl)

        if is_bin_neg:
            neg_key = bin_label_map_neg[lbl]
        else:
            neg_key = keys_neg.get(lbl)

        # Standard ex_only / im_only suppression (non-bin rows only)
        if not is_bin_pos and not is_bin_neg:
            if lbl in ex_only_labels:
                neg_key = None
            if lbl in im_only_labels:
                pos_key = None

        # Bin rows only appear in their own column
        # — unless both_bins_same_n, in which case both get same label
        #   and both keys should already be populated above
        if not both_bins_same_n:
            if is_bin_pos and not is_bin_neg:
                neg_key = None
            if is_bin_neg and not is_bin_pos:
                pos_key = None

        # For vhat, also look directly in params for bin-mode columns
        if lbl == r"$\hat{v}_{it}$":
            if has_bins_pos and pos_key is None:
                pos_key = _find_param(res_pos, ["vhat", "v_hat"])
            if has_bins_neg and neg_key is None:
                neg_key = _find_param(res_neg, ["vhat", "v_hat"])

        b_p, se_p, p_p = _coef_se_p(res_pos, pos_key)
        b_n, se_n, p_n = _coef_se_p(res_neg, neg_key)

        stars_p = _stars(p_p) if p_p is not None else ""
        stars_n = _stars(p_n) if p_n is not None else ""

        if same_line_se:
            se_str_p = f" ({_fmt_num(se_p, ndp)})" if se_p is not None else ""
            se_str_n = f" ({_fmt_num(se_n, ndp)})" if se_n is not None else ""
            cell_p = (_fmt_num(b_p, ndp) + stars_p + se_str_p
                      if b_p is not None else "")
            cell_n = (_fmt_num(b_n, ndp) + stars_n + se_str_n
                      if b_n is not None else "")
            lines.append(f"{lbl} & {cell_n.strip()} & {cell_p.strip()} \\\\")
            lines.append(r"\addlinespace[0.3em]")
        else:
            cell_p = (_fmt_num(b_p, ndp) + stars_p).strip()
            cell_n = (_fmt_num(b_n, ndp) + stars_n).strip()
            lines.append(f"{lbl} & {cell_n} & {cell_p} \\\\")
            se_cell_p = f"({_fmt_num(se_p, ndp)})" if se_p is not None else ""
            se_cell_n = f"({_fmt_num(se_n, ndp)})" if se_n is not None else ""
            lines.append(f" & {se_cell_n} & {se_cell_p} \\\\")
            lines.append(r"\addlinespace[0.5em]")

    lines.append(r"Entity and month FE & Yes & Yes \\")
    if not appendix_table:
        lines.append(r"Controls & Yes & Yes \\")
    lines.append(r"\midrule")

    # ------------------------------------------------------------------
    # Summary stats
    # ------------------------------------------------------------------
    n_pos = int(round(float(_get_attr(res_pos, "nobs", 0) or 0)))
    n_neg = int(round(float(_get_attr(res_neg, "nobs", 0) or 0)))
    lines.append(rf"Observations & {n_neg} & {n_pos} \\")

    r2_pos_s = _fmt_num(res_pos.prsquared, 3) if res_pos.prsquared is not None else ""
    r2_neg_s = _fmt_num(res_neg.prsquared, 3) if res_neg.prsquared is not None else ""
    lines.append(rf"Pseudo $R^{{2}}$ & {r2_neg_s} & {r2_pos_s} \\")

    # ------------------------------------------------------------------
    # First-stage statistics
    # ------------------------------------------------------------------
    if include_first_stage and (pos_has_fs or neg_has_fs):
        lines.append(r"\midrule")

        fsF_pos = (_joint_f_stat(fs_pos, first_stage_joint_restriction,
                                 include_p=include_joint_p)
                   if pos_has_fs else "")
        fsF_neg = (_joint_f_stat(fs_neg, first_stage_joint_restriction,
                                 include_p=include_joint_p)
                   if neg_has_fs else "")

        lines.append(rf"First-stage joint $F$-stat & {fsF_neg} & {fsF_pos} \\")

        fs_r2_pos_v = _get_attr(fs_pos, "rsquared") if pos_has_fs else None
        fs_r2_neg_v = _get_attr(fs_neg, "rsquared") if neg_has_fs else None
        fs_r2_pos_s = _fmt_num(fs_r2_pos_v, ndp_fs_r2) if fs_r2_pos_v is not None else ""
        fs_r2_neg_s = _fmt_num(fs_r2_neg_v, ndp_fs_r2) if fs_r2_neg_v is not None else ""
        lines.append(rf"First-stage $R^{{2}}$ & {fs_r2_neg_s} & {fs_r2_pos_s} \\")

    # ------------------------------------------------------------------
    # Close table
    # ------------------------------------------------------------------
    lines += [r"\bottomrule", r"\end{tabular}"]

    if slides:
        lines += [
            r"\end{adjustbox}",
            r"\vspace{0.2em}",
            r"{\tiny " + se_note_text + r"\\",
        ]
        if has_bins_pos or has_bins_neg:
            lines.append(r"* p $<$ 0.10, ** p $<$ 0.05, *** p $<$ 0.01}")
    else:
        lines += [
            r"\begin{tablenotes}[flushleft]",
            rf"\item {se_note_text}.",
        ]
        if not appendix_table:
            lines.append(
                r"\item Controls include contemporaneous values and lags of "
                r"wind, solar, and load, lags of neighboring wind, solar, and "
                r"load, as well as carbon, gas, and coal prices. "
                r"Full coefficient estimates are reported in "
                r"Table~\ref{tab:iv_2sri_appendix}."
            )
        lines += [
            r"\item * p $<$ 0.10, ** p $<$ 0.05, *** p $<$ 0.01",
            r"\end{tablenotes}",
            r"\end{threeparttable}",
            r"\end{adjustbox}",
            r"\end{table}",
        ]

    return "\n".join(lines)

# Load regression frame

In [ ]:
df_mod = pd.read_pickle('../data/df_preprocessed.pkl')

In [ ]:
df_mod_pos = df_mod.loc[
    (df_mod["competitive_dispatch_status_discretized"] == 0)].copy() 

df_mod_neg = df_mod.loc[
    (df_mod["competitive_dispatch_status_discretized"] == 1)].copy() 

# Keep only entities with variation in outcome
y_var = df_mod_pos.groupby("EIC")["deviation_discretized"].nunique()
valid_entities = y_var[y_var > 1].index
df_mod_pos = df_mod_pos[df_mod_pos["EIC"].isin(valid_entities)].copy()

y_var = df_mod_neg.groupby("EIC")["deviation_discretized"].nunique()
valid_entities = y_var[y_var > 1].index
df_mod_neg = df_mod_neg[df_mod_neg["EIC"].isin(valid_entities)].copy()

# Entity dummies
df_mod_pos_with_ee = pd.get_dummies(df_mod_pos, columns=["EIC"], prefix="entity", drop_first=True)
df_mod_neg_with_ee = pd.get_dummies(df_mod_neg, columns=["EIC"], prefix="entity", drop_first=True)

y_pos = (df_mod_pos_with_ee["deviation_discretized"] == 1).astype(int)
y_neg = (df_mod_neg_with_ee["deviation_discretized"] == -1).astype(int)

In [ ]:
eic_to_name = {
    '11WD2FARG0001262': 'Farge',
    '11WD2GE2D000056A': 'Franken 1 B2',
    '11WD2GEB1000055R': 'Franken 1 B1',
    '11WD2GKBK000335V': 'GKB Bremen',
    '11WD2HUNT0000644': 'Huntorf',
    '11WD2IRG4000061X': 'Irsching 4',
    '11WD2MEHR2CSWHCZ': 'Mehrum C',
    '11WD2WILH0001255': 'Wilhelmshaven',
    '11WD7BERG1S--A-X': 'Bergkamen A',
    '11WD7GERS-G-BLIY': 'Gersteinwerk I',
    '11WD7GERS5S-K1-J': 'Gersteinwerk K1',
    '11WD7HERD2G-H6-X': 'Cuno Herdecke H6',
    '11WD7KWHU1GBL10E': 'Trianel Hamm B10',
    '11WD7KWHU1GBL20B': 'Trianel Hamm B20',
    '11WD7KWKN-20-EEW': 'Knapsack 20',
    '11WD7KWKN-KW-EEN': 'Knapsack KW',
    '11WD7MITB1C---A1': 'Bexbach 1',
    '11WD7NEUR1B--C-U': 'Neurath C',
    '11WD7NEUR1B--F-L': 'Neurath F',
    '11WD7NEUR1B--G-I': 'Neurath G',
    '11WD7NIED2B--E-B': 'Niederaußem E',
    '11WD7WEIS1B--F-M': 'Weisweiler F',
    '11WD7WEIS5GVGTGU': 'Weisweiler VGT G3',
    '11WD7WEIS5GVGTHS': 'Weisweiler VGT H3',
    '11WD8BOXB1L---R0': 'Boxberg R',
}

In [ ]:
RESULTS_DIR = Path("results_cache")
RESULTS_DIR.mkdir(exist_ok=True)

# Logit IV (2SRI)

## y=withholding, x=import headroom

In [ ]:
out_neg_sq = run_model("im_headroom", 
                n_boot=500, 
                quadratic_headroom=True,
                fs_entity_fe=True,
                fs_month_fe=True,
                add_lags=2)

In [ ]:
save_result(out_neg_sq, "out_neg_sq")

In [ ]:
out_neg_sq = load_result("out_neg_sq")

In [ ]:
b1 = out_neg_sq["results_table"].loc["im_headroom", "coef"]
b2 = out_neg_sq["results_table"].loc["im_headroom_sq", "coef"]
H  = out_neg_sq["all_data"]["im_headroom"].median()
slope = b1 + 2*b2*H            # d log-odds / dH
pct = (np.exp(-slope) - 1) * 100   # % odds change per 1 GW REDUCTION
print(f"H={H:.2f} GW, slope={slope:.4f}, per-GW odds change={pct:.1f}%")

In [ ]:
b1 = out_neg_sq["results_table"].loc["im_headroom", "coef"]
b2 = out_neg_sq["results_table"].loc["im_headroom_sq", "coef"]
H  = out_neg_sq["all_data"]["im_headroom"].quantile(0.05)
slope = b1 + 2*b2*H            # d log-odds / dH
pct = (np.exp(-slope) - 1) * 100   # % odds change per 1 GW REDUCTION
print(f"H={H:.2f} GW, slope={slope:.4f}, per-GW odds change={pct:.1f}%")

In [ ]:
b1 = out_neg_sq["results_table"].loc["im_headroom", "coef"]
b2 = out_neg_sq["results_table"].loc["im_headroom_sq", "coef"]
h0  = out_neg_sq["all_data"]["im_headroom"].quantile(0.05)

d_eta = -(b1 + 2*b2*h0)

for p0 in (0.30, 0.50, 0.70):
    eta0 = np.log(p0/(1-p0))
    p1   = 1/(1+np.exp(-(eta0 + d_eta)))
    print(f"p0={p0:.2f}  ->  p1={p1:.3f}   (Δ = {(p1-p0)*100:+.2f} pp)")

## y=withholding, x=import headroom (decile bin specification)

In [ ]:
out_neg_bins = run_model(
    x_col="im_headroom",
    bin_dummies=True,
    n_bins=10,
    bin_reference=4,
    fs_entity_fe=True,
    fs_month_fe=True,
    n_boot=5,
    add_lags=2,
)

In [ ]:
save_result(out_neg_bins, "out_neg_bins")

In [ ]:
out_neg_bins = load_result("out_neg_bins")

## y=push-in, x=export headroom

In [ ]:
out_pos_sq = run_model("ex_headroom", 
                n_boot=500, 
                quadratic_headroom=True, 
                fs_entity_fe=True, 
                fs_month_fe=True, 
                add_lags=2)

In [ ]:
save_result(out_pos_sq, "out_pos_sq")

In [ ]:
out_pos_sq = load_result("out_pos_sq")

In [ ]:
b1 = out_pos_sq["results_table"].loc["ex_headroom", "coef"]
b2 = out_pos_sq["results_table"].loc["ex_headroom_sq", "coef"]
H  = out_pos_sq["all_data"]["ex_headroom"].median() 
slope = b1 + 2*b2*H            # d log-odds / dH
pct = (np.exp(-slope) - 1) * 100   # % odds change per 1 GW REDUCTION
print(f"H={H:.2f} GW, slope={slope:.4f}, per-GW odds change={pct:.1f}%")

In [ ]:
b1 = out_pos_sq["results_table"].loc["ex_headroom", "coef"]
b2 = out_pos_sq["results_table"].loc["ex_headroom_sq", "coef"]
H = out_pos_sq["all_data"]["ex_headroom"].quantile(0.05) 
slope = b1 + 2*b2*H            # d log-odds / dH
pct = (np.exp(-slope) - 1) * 100   # % odds change per 1 GW REDUCTION
print(f"H={H:.2f} GW, slope={slope:.4f}, per-GW odds change={pct:.1f}%")

In [ ]:
b1 = out_pos_sq["results_table"].loc["ex_headroom", "coef"]
b2 = out_pos_sq["results_table"].loc["ex_headroom_sq", "coef"]
h0 = out_pos_sq["all_data"]["ex_headroom"].quantile(0.05)

d_eta = -(b1 + 2*b2*h0)

for p0 in (0.30, 0.50, 0.70):
    eta0 = np.log(p0/(1-p0))
    p1   = 1/(1+np.exp(-(eta0 + d_eta)))
    print(f"p0={p0:.2f}  ->  p1={p1:.3f}   (Δ = {(p1-p0)*100:+.2f} pp)")

## y=push-in, x=export headroom (decile bin specification)

In [ ]:
out_pos_bins = run_model(
    x_col="ex_headroom",
    bin_dummies=True,
    n_bins=10,
    bin_reference=4,     
    fs_entity_fe=True,
    fs_month_fe=True,
    n_boot=500,
    add_lags=2,
    include_interaction=False,
)

In [ ]:
save_result(out_pos_bins, "out_pos_bins")

In [ ]:
out_pos_bins = load_result("out_pos_bins")

# Plain logit

## y=withholding, x=import headroom

In [ ]:
out_neg_plain = run_model("im_headroom",
    n_boot=0,
    plain_logit=True, 
    fs_entity_fe=True, 
    fs_month_fe=True, 
    quadratic_headroom=True, 
    add_lags=2)

In [ ]:
save_result(out_neg_plain, "out_neg_plain")

In [ ]:
out_neg_plain = load_result("out_neg_plain")

## y=push-in, x=export headroom

In [ ]:
out_pos_plain = run_model("ex_headroom", 
    n_boot=0, 
    plain_logit=True, 
    fs_entity_fe=True, 
    fs_month_fe=True, 
    quadratic_headroom=True, 
    add_lags=2)

In [ ]:
save_result(out_pos_plain, "out_pos_plain")

In [ ]:
out_pos_plain = load_result("out_pos_plain")

# Tables

### Table 1

In [ ]:
latex_table1 = make_summary_stats_latex(df_mod, df_mod_neg, df_mod_pos)
print(latex_table1)

### Table 2

In [ ]:
latex_table2 = make_iv_2sri_latex_table(
    out_neg_sq, out_pos_sq,
    caption="2SRI regression results",
    label="tab:iv_2sri",
)

print(latex_table2)

### Table A1

In [ ]:
latex_table_a1 = make_iv_2sri_latex_table(
    out_neg_sq, out_pos_sq,
    caption="Full 2SRI regression results",
    label="tab:iv_2sri_appendix",
    appendix_table=True,
)

print(latex_table_a1)

### Table A2

In [ ]:
latex_table_a2 = make_iv_2sri_latex_table(
    out_neg_bins, out_pos_bins,
    caption="2SRI regression results: decile bin specification",
    label="tab:iv_2sri_bin",
)

print(latex_table_a2)

# Figures

### Figure 3

In [ ]:
plt.rcParams.update({
    "font.size": 12,
    "axes.titlesize": 14,
    "axes.labelsize": 13,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "legend.fontsize": 11,
})

dtcol = "DateTime"
df_mod[dtcol] = pd.to_datetime(df_mod[dtcol], utc=True, errors="coerce")

cols = ["price", "min_np", "max_np", "net_pos"]

t0 = df_mod[dtcol].min()
t1 = t0 + pd.Timedelta(days=7)

week = df_mod.loc[df_mod[dtcol].between(t0, t1, inclusive="left"), [dtcol] + cols].dropna(subset=[dtcol])

week_ts = (
    week.groupby(dtcol, as_index=True)[cols]
        .mean()
        .sort_index()
)

week_ts[["min_np", "max_np", "net_pos"]] /= 1000.0

fig, ax = plt.subplots(figsize=(10, 4.5))
ax2 = ax.twinx()

band = ax.fill_between(
    week_ts.index,
    week_ts["min_np"].to_numpy(),
    week_ts["max_np"].to_numpy(),
    alpha=0.2,
    label="Min/Max net position bounds (GW)",
)
ln1 = ax.plot(week_ts.index, week_ts["net_pos"], alpha=0.7, label="Net position (GW)")

ax.set_ylabel("Net position (GW)")
ax.grid(True, alpha=0.3)

ln2 = ax2.plot(
    week_ts.index,
    week_ts["price"],
    alpha=0.7,
    color="tab:green",
    label="Day-ahead price (€/MWh)",
)
ax2.set_ylabel("Day-ahead price (€/MWh)")

handles = [band] + ln1 + ln2
labels = [h.get_label() for h in handles]
ax.legend(handles, labels, loc="lower right")

ax.xaxis.set_major_locator(mdates.DayLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m-%d"))
fig.autofmt_xdate()

plt.tight_layout()
fig.savefig("../figures/figure_3.pdf")
plt.show()

### Figure 4

In [ ]:
plt.rcParams.update({
    "font.size": 11,        # base
    "axes.titlesize": 13,
    "axes.labelsize": 12,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 10,
})

im = df_mod["im_headroom"].dropna().to_numpy()
ex = df_mod["ex_headroom"].dropna().to_numpy()

data = np.concatenate([im, ex])

fig, ax = plt.subplots(figsize=(6, 4.5))

bins = np.arange(np.floor(data.min() * 5) / 5,
                 np.ceil(data.max() * 5) / 5 + 0.2,
                 0.2)
ax.hist(im, bins=bins, density=False, alpha=0.6, label="Import headroom")
ax.hist(ex, bins=bins, density=False, alpha=0.6, label="Export headroom")

ax.set_ylabel("Number of observations")
ax.set_xlabel("Headroom (GW)")
ax.legend(frameon=False)

plt.tight_layout()
fig.savefig("../figures/figure_4.pdf")
plt.show()


### Figure 7


In [ ]:
fig = dose_response_plot(out_neg_sq, out_pos_sq, x_axis="gw", center_at_median=True)
fig.savefig("..//figures//figure_7.pdf", bbox_inches="tight")

### Figure 8


In [ ]:
fig = dose_response_plot(
    out_neg_sq, out_pos_sq,
    overlay_plain=True,
    out_neg_plain=out_neg_plain,
    out_pos_plain=out_pos_plain,
    x_axis="gw",
    center_at_median=True
)
fig.savefig("..//figures//figure_8.pdf", bbox_inches="tight")

### Figure 9


In [ ]:
fig, axes = plot_bin_marginal_effects(
    out_neg_bins=out_neg_bins,
    out_pos_bins=out_pos_bins,
    save_path="../figures/figure_9.pdf",
    dpi=300,
)

### Figure A1

In [ ]:
raw_withhold = get_entity_raw_probs(out_neg_sq)
raw_pushin   = get_entity_raw_probs(out_pos_sq)

entities_both_raw = raw_withhold.index.intersection(raw_pushin.index)
raw_df = pd.DataFrame({
    'withhold_rate': raw_withhold[entities_both_raw],
    'pushin_rate':   raw_pushin[entities_both_raw],
})

entity_types_raw = df_mod.groupby('EIC')['production_type'].first()
raw_df = raw_df.merge(entity_types_raw, left_index=True, right_index=True, how='left')

colors_raw = {
    'Fossil Hard coal': '#2C3E50',
    'Fossil Gas gas/steam turbine': '#E74C3C',
    'Fossil Gas Combined cycle': '#3498DB',
    'Fossil Brown coal/Lignite': '#7D6608',
}

fig_raw, ax_raw = plt.subplots(figsize=(5, 5))

for ptype in raw_df['production_type'].dropna().unique():
    mask = raw_df['production_type'] == ptype
    ax_raw.scatter(raw_df.loc[mask, 'withhold_rate'],
                   raw_df.loc[mask, 'pushin_rate'],
                   c=colors_raw.get(ptype, 'lightgray'),
                   label=short_label_raw(ptype),
                   alpha=0.7, s=100, edgecolors='black', linewidth=0.5)

if raw_df['production_type'].isna().any():
    mask = raw_df['production_type'].isna()
    ax_raw.scatter(raw_df.loc[mask, 'withhold_rate'],
                   raw_df.loc[mask, 'pushin_rate'],
                   c='lightgray', label='Unknown',
                   alpha=0.5, s=100, edgecolors='black', linewidth=0.5)

max_val_raw = max(raw_df['withhold_rate'].max(), raw_df['pushin_rate'].max())
ax_raw.plot([0, max_val_raw], [0, max_val_raw], 'k--', alpha=0.3, linewidth=1,
            label='Equal rates of deviation')

ax_raw.set_xlabel('Average rate of negative deviation', fontsize=12)
ax_raw.set_ylabel('Average rate of positive deviation', fontsize=12)
ax_raw.set_xlim(0, None)
ax_raw.set_ylim(0, None)
ax_raw.grid(alpha=0.3)

corr_raw = raw_df[['withhold_rate', 'pushin_rate']].corr().iloc[0, 1]
ax_raw.text(0.65, 0.55, f"Correlation: {corr_raw:.2f}",
            transform=ax_raw.transAxes, fontsize=10,
            verticalalignment='top',
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

ax_raw.legend(loc='upper right', frameon=True, framealpha=0.9, fontsize=9)

plt.tight_layout()
plt.savefig('../figures/figure_a1.pdf', dpi=300, bbox_inches='tight')
plt.show()

### Figure A2

In [ ]:
DEFAULT_TYPE_COLORS = {
    "Fossil Hard coal":            "#2C3E50",   
    "Fossil Gas gas/steam turbine": "#E74C3C",  
    "Fossil Gas Combined cycle":    "#3498DB",  
    "Fossil Brown coal/Lignite":    "#7D6608", 
}
_UNKNOWN_COLOR = "lightgray"

In [ ]:
kw = dict(quadratic_headroom=True, include_interaction=False,
          add_lags=2, fs_entity_fe=True, fs_month_fe=True, n_boot=200)

eic_codes_neg = df_mod_neg['EIC'].unique()
bundles_neg, h_sample_neg = leave_one_out_draws("im_headroom", eic_codes_neg, **kw)
with open("loo_neg_bundles.pkl", "wb") as f:
    pickle.dump((bundles_neg, h_sample_neg), f)

eic_codes_pos = df_mod_pos['EIC'].unique()
bundles_pos, h_sample_pos = leave_one_out_draws("ex_headroom", eic_codes_pos, **kw)
with open("loo_pos_bundles.pkl", "wb") as f:
    pickle.dump((bundles_pos, h_sample_pos), f)

for tag, bundles, h_sample in [("neg", bundles_neg, h_sample_neg),
                               ("pos", bundles_pos, h_sample_pos)]:
    h10 = float(np.quantile(h_sample, 0.10))
    h50 = float(np.quantile(h_sample, 0.50))
    loo_table(bundles, h_eval=h50).to_pickle(f"loo_{tag}.pkl")    
    loo_table(bundles, h_eval=h10).to_pickle(f"loo_{tag}_p10.pkl")

In [ ]:
with open("loo_neg_bundles.pkl", "rb") as f:
    bundles_neg, h_sample_neg = pickle.load(f)
with open("loo_pos_bundles.pkl", "rb") as f:
    bundles_pos, h_sample_pos = pickle.load(f)

pct = 0.10
loo_neg = loo_table(bundles_neg, h_eval=float(np.quantile(h_sample_neg, pct)))
loo_pos = loo_table(bundles_pos, h_eval=float(np.quantile(h_sample_pos, pct)))

fig = leave_one_out_plot(loo_neg, loo_pos, pct=pct, eic_to_name=eic_to_name)
fig.savefig("..//figures//figure_a2.pdf", bbox_inches="tight")